In [ ]:
#AINL

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import os
import re
import random

def clean_text(text):
    """Заменяем переносы строк и табуляции на пробелы"""
    if isinstance(text, str):
        text = re.sub(r'\n|\r|\t', ' ', text)
        text = re.sub(r'\s+', ' ', text)
        return text.strip()
    return text

# Чтение первого файла
df1 = pd.read_csv('train-kenlm.csv', usecols=['text', 'label'])

# Очистка текста
df1['text'] = df1['text'].fillna('').apply(clean_text)

# Фильтрация по длине (мин. 50 слов)
df1['word_count'] = df1['text'].str.split().str.len()
df1 = df1[df1['word_count'] >= 50].drop(columns='word_count').reset_index(drop=True)

# Разделение на human и ai
df1_human = df1[df1['label'] == 'human']
df1_ai = df1[df1['label'] != 'human']

print(f"Первый файл - после фильтрации:")
print(f"  Human: {len(df1_human)} текстов")
print(f"  AI: {len(df1_ai)} текстов")

# Балансировка для разделения
n_samples = min(len(df1_human), len(df1_ai))
print(f"\nБалансировка: берем {n_samples} текстов каждого класса")

if len(df1_human) > n_samples:
    df1_human = df1_human.sample(n_samples, random_state=42)
if len(df1_ai) > n_samples:
    df1_ai = df1_ai.sample(n_samples, random_state=42)

human_texts = df1_human['text'].tolist()
ai_texts = df1_ai['text'].tolist()

# Разделение 80/20 (80% обучение, 20% валидация)
human_train, human_val = train_test_split(human_texts, test_size=0.2, random_state=42, shuffle=True)
ai_train, ai_val = train_test_split(ai_texts, test_size=0.2, random_state=42, shuffle=True)

print(f"\nРазделение 80/20:")
print(f"  Обучение: human={len(human_train)}, ai={len(ai_train)}")
print(f"  Валидация: human={len(human_val)}, ai={len(ai_val)}")

# Сохраняем файлы обучения
with open('train_texts.txt', 'w', encoding='utf-8') as f:
    for text in human_train + ai_train:
        f.write(clean_text(text) + '\n')

# Сохраняем валидационные файлы
with open('val_human.txt', 'w', encoding='utf-8') as f:
    for text in human_val:
        f.write(clean_text(text) + '\n')

with open('val_ai.txt', 'w', encoding='utf-8') as f:
    for text in ai_val:
        f.write(clean_text(text) + '\n')

print("\nСохранено:")
print(f"  train_texts.txt: {len(human_train)+len(ai_train)} текстов")
print(f"  val_human.txt: {len(human_val)} текстов")
print(f"  val_ai.txt: {len(ai_val)} текстов")

# Запоминаем размер валидационной выборки для второго файла
val_size = len(human_val)
print(f"\nРазмер валидационной выборки: {val_size} текстов каждого класса")

Первый файл - после фильтрации:
  Human: 7275 текстов
  AI: 23963 текстов

Балансировка: берем 7275 текстов каждого класса

Разделение 80/20:
  Обучение: human=5820, ai=5820
  Валидация: human=1455, ai=1455

Сохранено:
  train_texts.txt: 11640 текстов
  val_human.txt: 1455 текстов
  val_ai.txt: 1455 текстов

Размер валидационной выборки: 1455 текстов каждого класса


In [2]:
# Чтение второго файла
df2 = pd.read_csv('train-kenlm.csv', usecols=['text', 'label'])

# Очистка текста
df2['text'] = df2['text'].fillna('').apply(clean_text)

# Фильтрация по длине (мин. 50 слов)
df2['word_count'] = df2['text'].str.split().str.len()
df2 = df2[df2['word_count'] >= 50].drop(columns='word_count').reset_index(drop=True)

# Разделение на human и ai
df2_human = df2[df2['label'] == 'human']
df2_ai = df2[df2['label'] != 'human']

print(f"Второй файл - после фильтрации:")
print(f"  Human: {len(df2_human)} текстов")
print(f"  AI: {len(df2_ai)} текстов")

# Берем сбалансированную выборку такого же размера как валидационная
n_samples = val_size  # размер из первой ячейки

if len(df2_human) < n_samples:
    print(f"⚠️ Внимание: human текстов меньше ({len(df2_human)}), берем все")
    n_samples = len(df2_human)

if len(df2_ai) < n_samples:
    print(f"⚠️ Внимание: ai текстов меньше ({len(df2_ai)}), берем все")
    n_samples = min(n_samples, len(df2_ai))

print(f"\nБерем {n_samples} текстов каждого класса")

# Случайная выборка
test_human = df2_human.sample(n_samples, random_state=42)['text'].tolist()
test_ai = df2_ai.sample(n_samples, random_state=42)['text'].tolist()

print(f"\nТестовая выборка:")
print(f"  Human: {len(test_human)} текстов")
print(f"  AI: {len(test_ai)} текстов")

# Сохраняем тестовые файлы
with open('test_human.txt', 'w', encoding='utf-8') as f:
    for text in test_human:
        f.write(clean_text(text) + '\n')

with open('test_ai.txt', 'w', encoding='utf-8') as f:
    for text in test_ai:
        f.write(clean_text(text) + '\n')

print("\nСохранено:")
print(f"  test_human.txt: {len(test_human)} текстов")
print(f"  test_ai.txt: {len(test_ai)} текстов")

Второй файл - после фильтрации:
  Human: 7275 текстов
  AI: 23963 текстов

Берем 1455 текстов каждого класса

Тестовая выборка:
  Human: 1455 текстов
  AI: 1455 текстов

Сохранено:
  test_human.txt: 1455 текстов
  test_ai.txt: 1455 текстов


In [3]:
#LLMTRACE

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import os
import re
import random

def clean_text(text):
    """Заменяем переносы строк и табуляции на пробелы"""
    if isinstance(text, str):
        text = re.sub(r'\n|\r|\t', ' ', text)
        text = re.sub(r'\s+', ' ', text)
        return text.strip()
    return text

# Определяем колонки для первого файла
cols = ['text', 'label', 'model'] if 'model' in pd.read_csv(
    'llmtrace_news.csv', nrows=0).columns else ['text', 'label']

df1 = pd.read_csv('llmtrace_news.csv', usecols=cols)

# Очистка текста
df1['text'] = df1['text'].fillna('').apply(clean_text)

# Фильтрация по длине (мин. 50 слов)
df1['word_count'] = df1['text'].str.split().str.len()
df1 = df1[df1['word_count'] >= 50].drop(columns='word_count').reset_index(drop=True)

# Разделение на human и ai
df1_human = df1[df1['label'] == 'human']
df1_ai = df1[df1['label'] != 'human']

print(f"Первый файл - после фильтрации по длине:")
print(f"  Human: {len(df1_human)} текстов")
print(f"  AI: {len(df1_ai)} текстов")

# Фильтрация AI по моделям (>=700 текстов)
if 'model' in df1.columns:
    model_counts = df1_ai['model'].value_counts()
    models_to_keep = model_counts[model_counts >= 700].index
    df1_ai = df1_ai[df1_ai['model'].isin(models_to_keep)]
    print(f"\nПосле фильтрации по моделям (>=700):")
    print(f"  AI: {len(df1_ai)} текстов")
    print(f"  Модели: {list(models_to_keep)}")

# Балансировка для разделения
n_samples = min(len(df1_human), len(df1_ai))
print(f"\nБалансировка: берем {n_samples} текстов каждого класса")

if len(df1_human) > n_samples:
    df1_human = df1_human.sample(n_samples, random_state=42)
if len(df1_ai) > n_samples:
    df1_ai = df1_ai.sample(n_samples, random_state=42)

human_texts = df1_human['text'].tolist()
ai_texts = df1_ai['text'].tolist()

# Разделение 80/20 (80% обучение, 20% валидация)
human_train, human_val = train_test_split(human_texts, test_size=0.2, random_state=42, shuffle=True)
ai_train, ai_val = train_test_split(ai_texts, test_size=0.2, random_state=42, shuffle=True)

print(f"\nРазделение 80/20:")
print(f"  Обучение: human={len(human_train)}, ai={len(ai_train)}")
print(f"  Валидация: human={len(human_val)}, ai={len(ai_val)}")

# Сохраняем файлы обучения
with open('train_texts.txt', 'w', encoding='utf-8') as f:
    for text in human_train + ai_train:
        f.write(clean_text(text) + '\n')

# Сохраняем валидационные файлы
with open('val_human.txt', 'w', encoding='utf-8') as f:
    for text in human_val:
        f.write(clean_text(text) + '\n')

with open('val_ai.txt', 'w', encoding='utf-8') as f:
    for text in ai_val:
        f.write(clean_text(text) + '\n')

print("\nСохранено:")
print(f"  train_texts.txt: {len(human_train)+len(ai_train)} текстов")
print(f"  val_human.txt: {len(human_val)} текстов")
print(f"  val_ai.txt: {len(ai_val)} текстов")

# Запоминаем размер валидационной выборки для второго файла
val_size = len(human_val)
print(f"\nРазмер валидационной выборки: {val_size} текстов каждого класса")

Первый файл - после фильтрации по длине:
  Human: 11276 текстов
  AI: 12610 текстов

После фильтрации по моделям (>=700):
  AI: 9196 текстов
  Модели: ['gpt-3.5', 'CohereForAI/c4ai-command-r-08-2024', 'yandex/YandexGPT-5-Lite-8B-instruct', 'unsloth/Llama-3.3-70B-Instruct', 'google/gemma-2-27b-it', '01-ai/Yi-1.5-34B-Chat', 'WizardLM-2-7B', 'Qwen/QwQ-32B', 'databricks/dbrx-instruct', 'Qwen/Qwen2.5-72B-Instruct', 'Meta-Llama-3-8B-Instruct', 'mistralai/Ministral-8B-Instruct-2410']

Балансировка: берем 9196 текстов каждого класса

Разделение 80/20:
  Обучение: human=7356, ai=7356
  Валидация: human=1840, ai=1840

Сохранено:
  train_texts.txt: 14712 текстов
  val_human.txt: 1840 текстов
  val_ai.txt: 1840 текстов

Размер валидационной выборки: 1840 текстов каждого класса


In [6]:
# Определяем колонки для второго файла
cols = ['text', 'label', 'model'] if 'model' in pd.read_csv(
    'llmtrace_reviews.csv', nrows=0).columns else ['text', 'label']

df2 = pd.read_csv('llmtrace_reviews.csv', usecols=cols)

# Очистка текста
df2['text'] = df2['text'].fillna('').apply(clean_text)

# Фильтрация по длине (мин. 50 слов)
df2['word_count'] = df2['text'].str.split().str.len()
df2 = df2[df2['word_count'] >= 50].drop(columns='word_count').reset_index(drop=True)

# Разделение на human и ai
df2_human = df2[df2['label'] == 'human']
df2_ai = df2[df2['label'] != 'human']

print(f"Второй файл - после фильтрации по длине:")
print(f"  Human: {len(df2_human)} текстов")
print(f"  AI: {len(df2_ai)} текстов")

# Фильтрация AI по моделям (>=700 текстов)
if 'model' in df2.columns:
    model_counts = df2_ai['model'].value_counts()
    models_to_keep = model_counts[model_counts >= 700].index
    df2_ai = df2_ai[df2_ai['model'].isin(models_to_keep)]
    print(f"\nПосле фильтрации по моделям (>=700):")
    print(f"  AI: {len(df2_ai)} текстов")
    print(f"  Модели: {list(models_to_keep)}")

# Берем сбалансированную выборку такого же размера как валидационная
n_samples = val_size

if len(df2_human) < n_samples:
    print(f"\n⚠️ Внимание: human текстов меньше ({len(df2_human)}), берем все")
    n_samples = len(df2_human)

if len(df2_ai) < n_samples:
    print(f"\n⚠️ Внимание: ai текстов меньше ({len(df2_ai)}), берем все")
    n_samples = min(n_samples, len(df2_ai))

print(f"\nБерем {n_samples} текстов каждого класса")

# Случайная выборка
test_human = df2_human.sample(n_samples, random_state=42)['text'].tolist()
test_ai = df2_ai.sample(n_samples, random_state=42)['text'].tolist()

print(f"\nТестовая выборка:")
print(f"  Human: {len(test_human)} текстов")
print(f"  AI: {len(test_ai)} текстов")

# Сохраняем тестовые файлы
with open('test_human.txt', 'w', encoding='utf-8') as f:
    for text in test_human:
        f.write(clean_text(text) + '\n')

with open('test_ai.txt', 'w', encoding='utf-8') as f:
    for text in test_ai:
        f.write(clean_text(text) + '\n')

print("\nСохранено:")
print(f"  test_human.txt: {len(test_human)} текстов")
print(f"  test_ai.txt: {len(test_ai)} текстов")

Второй файл - после фильтрации по длине:
  Human: 9072 текстов
  AI: 16641 текстов

После фильтрации по моделям (>=700):
  AI: 13331 текстов
  Модели: ['gpt-3.5', 'CohereForAI/c4ai-command-r-08-2024', '01-ai/Yi-1.5-34B-Chat', 'google/gemma-2-27b-it', 'unsloth/Llama-3.3-70B-Instruct', 'Qwen/Qwen2.5-72B-Instruct', 'databricks/dbrx-instruct', 'yandex/YandexGPT-5-Lite-8B-instruct', 'Qwen/QwQ-32B', 'mistralai/Ministral-8B-Instruct-2410', 'GigaChat-Max', 'yagpt']

Берем 1840 текстов каждого класса

Тестовая выборка:
  Human: 1840 текстов
  AI: 1840 текстов

Сохранено:
  test_human.txt: 1840 текстов
  test_ai.txt: 1840 текстов


In [7]:
#HARD

In [8]:
import json
import os
import random
import re
import numpy as np
from sklearn.model_selection import train_test_split

def clean_text(text):
    """Заменяем переносы строк и табуляции на пробелы"""
    if isinstance(text, str):
        text = re.sub(r'\n|\r|\t', ' ', text)
        text = re.sub(r'\s+', ' ', text)
        return text.strip()
    return text

def load_json_texts(filepath, source_type):
    """
    Загружает тексты из JSON файла по значению поля 'source'
    source_type: 'human', 'ai', 'ai+rew' (paraphrased)
    """
    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    texts = []
    for item in data:
        if item.get('source') == source_type:
            text = clean_text(item.get('text', ''))
            if len(text.split()) >= 50:
                texts.append(text)
    return texts

# Параметры первого файла
base_path = 'ru_hard_dataset'
domain = 'essay'  # или 'news', 'scientific_texts', 'long_sc'

# Загружаем human и ai
human_file = os.path.join(base_path, domain, 'original_essay.json')
ai_file = os.path.join(base_path, domain, 'generated_essays.json')

human_texts = load_json_texts(human_file, 'human')
ai_texts = load_json_texts(ai_file, 'ai')

print(f"Первый файл ({domain}) - после фильтрации по длине:")
print(f"  Human: {len(human_texts)} текстов")
print(f"  AI: {len(ai_texts)} текстов")

# Балансировка
n_samples = min(len(human_texts), len(ai_texts))
print(f"\nБалансировка: берем {n_samples} текстов каждого класса")

if len(human_texts) > n_samples:
    human_texts = random.sample(human_texts, n_samples)
if len(ai_texts) > n_samples:
    ai_texts = random.sample(ai_texts, n_samples)

# Разделение 80/20 (80% обучение, 20% валидация)
human_train, human_val = train_test_split(human_texts, test_size=0.2, random_state=42, shuffle=True)
ai_train, ai_val = train_test_split(ai_texts, test_size=0.2, random_state=42, shuffle=True)

print(f"\nРазделение 80/20:")
print(f"  Обучение: human={len(human_train)}, ai={len(ai_train)}")
print(f"  Валидация: human={len(human_val)}, ai={len(ai_val)}")

# Сохраняем файлы обучения
with open('train_texts.txt', 'w', encoding='utf-8') as f:
    for text in human_train + ai_train:
        f.write(clean_text(text) + '\n')

# Сохраняем валидационные файлы
with open('val_human.txt', 'w', encoding='utf-8') as f:
    for text in human_val:
        f.write(clean_text(text) + '\n')

with open('val_ai.txt', 'w', encoding='utf-8') as f:
    for text in ai_val:
        f.write(clean_text(text) + '\n')

print("\nСохранено:")
print(f"  train_texts.txt: {len(human_train)+len(ai_train)} текстов")
print(f"  val_human.txt: {len(human_val)} текстов")
print(f"  val_ai.txt: {len(ai_val)} текстов")

# Запоминаем размер валидационной выборки для второго файла
val_size = len(human_val)
print(f"\nРазмер валидационной выборки: {val_size} текстов каждого класса")

Первый файл (essay) - после фильтрации по длине:
  Human: 480 текстов
  AI: 480 текстов

Балансировка: берем 480 текстов каждого класса

Разделение 80/20:
  Обучение: human=384, ai=384
  Валидация: human=96, ai=96

Сохранено:
  train_texts.txt: 768 текстов
  val_human.txt: 96 текстов
  val_ai.txt: 96 текстов

Размер валидационной выборки: 96 текстов каждого класса


In [10]:
import json
import os
import random
import re
import numpy as np

def clean_text(text):
    """Заменяем переносы строк и табуляции на пробелы"""
    if isinstance(text, str):
        text = re.sub(r'\n|\r|\t', ' ', text)
        text = re.sub(r'\s+', ' ', text)
        return text.strip()
    return text

def load_json_texts(filepath, source_type):
    """
    Загружает тексты из JSON файла по значению поля 'source'
    source_type: 'human', 'ai', 'ai+rew' (paraphrased)
    """
    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    texts = []
    for item in data:
        if item.get('source') == source_type:
            text = clean_text(item.get('text', ''))
            if len(text.split()) >= 50:
                texts.append(text)
    return texts

# Параметры второго файла (другой домен)
base_path = 'ru_hard_dataset'
domain2 = 'news'  # или 'scientific_texts', 'long_sc' и т.д.

# Загружаем human и ai из второго домена
human_file2 = os.path.join(base_path, domain2, 'original_news.json')
ai_file2 = os.path.join(base_path, domain2, 'generated_news.json')

human_texts2 = load_json_texts(human_file2, 'human')
ai_texts2 = load_json_texts(ai_file2, 'ai')

print(f"Второй файл ({domain2}) - после фильтрации по длине:")
print(f"  Human: {len(human_texts2)} текстов")
print(f"  AI: {len(ai_texts2)} текстов")

# Берем сбалансированную выборку такого же размера как валидационная
n_samples = val_size

if len(human_texts2) < n_samples:
    print(f"\n⚠️ Внимание: human текстов меньше ({len(human_texts2)}), берем все")
    n_samples = len(human_texts2)

if len(ai_texts2) < n_samples:
    print(f"\n⚠️ Внимание: ai текстов меньше ({len(ai_texts2)}), берем все")
    n_samples = min(n_samples, len(ai_texts2))

print(f"\nБерем {n_samples} текстов каждого класса")

# Случайная выборка
test_human = random.sample(human_texts2, n_samples) if len(human_texts2) >= n_samples else human_texts2
test_ai = random.sample(ai_texts2, n_samples) if len(ai_texts2) >= n_samples else ai_texts2

print(f"\nТестовая выборка:")
print(f"  Human: {len(test_human)} текстов")
print(f"  AI: {len(test_ai)} текстов")

# Сохраняем тестовые файлы
with open('test_human.txt', 'w', encoding='utf-8') as f:
    for text in test_human:
        f.write(clean_text(text) + '\n')

with open('test_ai.txt', 'w', encoding='utf-8') as f:
    for text in test_ai:
        f.write(clean_text(text) + '\n')

print("\nСохранено:")
print(f"  test_human.txt: {len(test_human)} текстов")
print(f"  test_ai.txt: {len(test_ai)} текстов")

Второй файл (news) - после фильтрации по длине:
  Human: 480 текстов
  AI: 480 текстов

Берем 96 текстов каждого класса

Тестовая выборка:
  Human: 96 текстов
  AI: 96 текстов

Сохранено:
  test_human.txt: 96 текстов
  test_ai.txt: 96 текстов
